# Parameter selection guide: the measurements behind `docs/parameter_selection_guide.md`

Reviewer 1, comment 1 of the SoftwareX submission. Every table in the parameter selection guide,
Section 2.3 of the manuscript and the five-split spread quoted in Section 3.2 come from this notebook.

**Procedure.** The paper's own: band models are fitted on 10 isolates and their evidence is
calibrated on 5 others. The conflict sign is learned on 5 defect-planted *select* isolates,
and every number is read on 5 *qc* isolates the configuration never saw. Defects come from
`pypasi.degrade`, planted before the scaling. `SPLIT_REPEAT` re-draws which isolates go where.

**Requirements.** `pypasi >= 0.5.5` and the Ho et al. `.npy` files in `Bacteria/`.
**Runtime** is about 40 minutes on two cores, most of it in Sections 2 and 3.

## 0 · Setup

Shared with `short_exposure.ipynb`. It reproduces the split and the defects of `softwarex_examples.ipynb` exactly.

In [1]:
# --- shared setup: the data, the patient-disjoint split and the defects of the paper ---
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
import pypasi
from pypasi import BandNegotiationClassifier, BandSet, ControlProfile, PLSDA, Triage, degrade
from pypasi.datasets import load_bacteria

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Control limits built from a random split")
assert tuple(int(p) for p in pypasi.__version__.split(".")[:3]) >= (0, 5, 5), "needs pypasi >= 0.5.5"

DATA_DIR = Path("Bacteria")
SEED = 0
# the configuration of softwarex_examples.ipynb; every function below defaults to it
N_BANDS, TOPOLOGY, REGIME, N_COMPONENTS, SCALE_NAME = 7, "complete", "H1", 10, "SNV"
CALIBRATION, CAL_SPLIT = "auto", 0.25
REVIEW_RATE, NOVELTY_Q = 0.10, 0.99
K_SIGMA, LASER_NM, PROFILE_STATS = 3.0, 633.0, ("mean_stress", "band_confidence")
BATCH_N, LOCALISE_REPEATS = 100, 6
SPECTRA_PER_PATIENT = 400


def minmax(X):
    X = np.asarray(X, float)
    lo, hi = X.min(1, keepdims=True), X.max(1, keepdims=True)
    return (X - lo) / np.where(hi > lo, hi - lo, 1.0)


def snv(X):
    X = np.asarray(X, float)
    s = X.std(1, keepdims=True)
    return (X - X.mean(1, keepdims=True)) / np.where(s > 0, s, 1.0)


SCALERS = {"SNV": snv, "min-max": lambda X: np.asarray(X, float)}

data = load_bacteria(DATA_DIR, normalize=False)
AXIS = data.axis
MM_all, y_all = minmax(data.X_dev), data.y_dev      # 2018 cohort, per-spectrum min-max
MM_B, y_B = minmax(data.X_test), data.y_test        # 2019 cohort (external)

# Patient = block of 400 consecutive spectra; verified, never assumed
groups = np.arange(len(y_all)) // SPECTRA_PER_PATIENT
for g in np.unique(groups):
    assert np.unique(y_all[groups == g]).size == 1, "row order is not grouped by patient"
N_PAT = groups.max() + 1
PAT_CLASS = np.array([y_all[groups == g][0] for g in range(N_PAT)])

BAND_RANGE = (400.0, min(1800.0, float(AXIS.max())))
REF_BANDS = BandSet.equal_width(AXIS, 7, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
REF_EDGES = [(b.lo, b.hi) for b in REF_BANDS.bands]   # where local defects are planted


def _take(pool, per_class, rng):
    taken = []
    for c in np.unique(PAT_CLASS[pool]):
        taken.extend(rng.choice(pool[PAT_CLASS[pool] == c], per_class, replace=False))
    taken = np.array(sorted(taken))
    return taken, np.setdiff1d(pool, taken)


def make_split(split_repeat=0):
    """The paper's split (fit 10 | limits 5 | select 5 | qc 5 isolates) and its defects."""
    rng = np.random.default_rng(SEED + split_repeat)
    pool = np.arange(N_PAT)
    p_qc, pool = _take(pool, 1, rng)
    p_sel, pool = _take(pool, 1, rng)
    p_lim, pool = _take(pool, 1, rng)
    rows = lambda p: np.flatnonzero(np.isin(groups, p))  # noqa: E731
    ix = {"fit": rows(pool), "lim": rows(p_lim), "sel": rows(p_sel), "qc": rows(p_qc)}
    ix["ref"] = np.sort(np.concatenate([ix["lim"], ix["sel"], ix["qc"]]))
    return {
        "repeat": split_repeat, "p_qc": p_qc, "ix": ix,
        "chart_patients": np.sort(np.concatenate([p_lim, p_sel])),
        "sel_def": degrade.plant(MM_all[ix["sel"]], AXIS, edges=REF_EDGES, random_state=SEED + 77),
        "qc_def": degrade.plant(MM_all[ix["qc"]], AXIS, edges=REF_EDGES, random_state=SEED),
    }


def binf(X, f):
    """Average f adjacent channels: a coarser detector sampling."""
    X = np.asarray(X, float)
    if f == 1:
        return X
    n = (X.shape[-1] // f) * f
    return X[..., :n].reshape(*X.shape[:-1], n // f, f).mean(-1)


def fit_model(sp, *, n_bands=N_BANDS, topology=TOPOLOGY, scale=SCALE_NAME, mode="equal",
              regime=REGIME, n_components=N_COMPONENTS, bin_factor=1):
    """Fit the band ensemble and its Triage exactly as the paper does, for one configuration."""
    S = SCALERS[scale]
    prep = lambda X: S(binf(X, bin_factor))  # noqa: E731
    ax = binf(AXIS, bin_factor)
    ix = sp["ix"]
    if mode == "equal":
        bands = BandSet.equal_width(ax, n_bands, lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    else:
        bands = BandSet.peak_informed(ax, binf(MM_all[ix["fit"]], bin_factor), n_bands,
                                      lo=BAND_RANGE[0], hi=BAND_RANGE[1], axis_name="cm-1")
    lim = (prep(MM_all[ix["lim"]]), y_all[ix["lim"]])
    t0 = time.perf_counter()
    model = BandNegotiationClassifier(
        bands=bands, base_estimator=PLSDA(n_components=n_components), topology=topology,
        regime=regime, calibration=CALIBRATION, calibration_data=lim,
        calibration_split=CAL_SPLIT, random_state=SEED).fit(prep(MM_all[ix["fit"]]), y_all[ix["fit"]])
    t_fit = time.perf_counter() - t0
    tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,
                 n_components=n_components).fit(
        model, *lim, orientation_data=(prep(sp["sel_def"].X), sp["sel_def"].degraded))
    return {"model": model, "triage": tri, "bands": bands, "prep": prep, "t_fit": t_fit}


def localise(sp, fm, planted_intervals=None, trials=None):
    """Plant a local defect in a batch of 100 spectra of one qc isolate; does the chart name it?

    A hit is a named band that overlaps the planted interval; chance is the share of bands
    that overlap it. By default the seven published intervals, six trials each (42 trials).
    """
    prep, bands, model = fm["prep"], fm["bands"], fm["model"]
    edges, labs = [(b.lo, b.hi) for b in bands.bands], bands.labels
    rngb = np.random.default_rng(SEED + 11)
    ref = [prep(MM_all[np.sort(rngb.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))])
           for g in sp["chart_patients"]]
    prof = ControlProfile.fit(model, batches=ref, k=K_SIGMA, laser_nm=LASER_NM,
                              statistics=PROFILE_STATS, random_state=SEED)
    if trials is None:
        trials = [(lo, hi, np.random.default_rng(500 * t + rep))
                  for t, (lo, hi) in enumerate(REF_EDGES) for rep in range(LOCALISE_REPEATS)]
    hits, chance = [], []
    for lo, hi, r in trials:
        sel = (AXIS >= lo) & (AXIS < hi)
        overlap = [j for j, (a, b) in enumerate(edges) if a < hi and b > lo]
        g = r.choice(sp["p_qc"])
        Xl = MM_all[np.sort(r.choice(np.flatnonzero(groups == g), BATCH_N, replace=False))].copy()
        Xl[:, sel] = Xl[:, sel] * 0.55 + r.normal(0, 0.02, (len(Xl), int(sel.sum())))
        tab = prof.check(prep(Xl), name="trial").table
        named = tab.assign(az=tab["z"].abs()).groupby("label")["az"].max().idxmax()
        hits.append(labs.index(named) in overlap)
        chance.append(len(overlap) / len(edges))
    return float(np.mean(hits)), float(np.mean(chance))


def evaluate(sp, *, loc=True, **config):
    """Every number the guide reports for one configuration on one split."""
    cfg = {"n_bands": N_BANDS, "topology": TOPOLOGY, "scale": SCALE_NAME, "mode": "equal",
           "regime": REGIME, "n_components": N_COMPONENTS, "bin_factor": 1, **config}
    fm = fit_model(sp, **cfg)
    ix, qd, prep = sp["ix"], sp["qc_def"], fm["prep"]
    y_qc = y_all[ix["qc"]]
    t0 = time.perf_counter()
    T = fm["triage"].assess(prep(qd.X), y_qc).table
    t_audit = time.perf_counter() - t0
    c = T["conflict"].to_numpy()
    fam = {k: roc_auc_score(qd.degraded[(qd.kind == k) | (qd.degraded == 0)],
                            c[(qd.kind == k) | (qd.degraded == 0)]) for k in degrade.PAPER_FAMILIES}
    row = {"repeat": sp["repeat"], **cfg,
           "channels_per_band": int(np.median([b.n_features for b in fm["bands"].bands])),
           "conflict_vs_degraded": roc_auc_score(qd.degraded, c),
           "conflict_vs_wrong": roc_auc_score(1 - T["correct"], c),
           "local": fam["local contamination"],
           "global_mean": float(np.mean([fam[k] for k in degrade.PAPER_FAMILIES if k != "local contamination"])),
           "acc_in_cohort": accuracy_score(y_all[ix["ref"]], fm["model"].predict(prep(MM_all[ix["ref"]]))),
           "acc_external": accuracy_score(y_B, fm["model"].predict(prep(MM_B))),
           "t_fit_s": fm["t_fit"], "t_audit_s": t_audit,
           "orientation_margin": fm["triage"].orientation_margin_}
    if loc:
        row["loc_hit"], row["loc_chance"] = localise(sp, fm)
    return row


In [2]:
def check(rows):
    """Compare this run with the values quoted in the guide and the response letter."""
    t = pd.DataFrame(rows, columns=["quantity", "this_run", "quoted", "tol"])
    t["ok"] = (t["this_run"] - t["quoted"]).abs() <= t["tol"]
    print(t.round(4).to_string(index=False))
    print("\nALL QUOTED VALUES REPRODUCE" if t["ok"].all() else
          f"\n{(~t['ok']).sum()} VALUES DO NOT REPRODUCE: correct the text or the run")
    return t

## 1 · One split, many configurations (split 0)

Band count, topology, scaling, regime and peak-informed bands. Split 0 is the split of the paper; Section 2 repeats the key rows on four more.

In [3]:
t0 = time.time()
sp0 = make_split(0)
GRID = ([dict(n_bands=k) for k in (3, 5, 7, 10, 14, 20, 28)]
        + [dict(n_bands=k, topology="knn") for k in (7, 28)]
        + [dict(n_bands=7, topology="chain"), dict(n_bands=7, scale="min-max")]
        + [dict(n_bands=7, regime=r) for r in ("plain", "H2")]
        + [dict(n_bands=k, mode="peak") for k in (5, 7, 10, 14)])
grid = pd.DataFrame([evaluate(sp0, **g) for g in GRID])
cols = ["n_bands", "topology", "scale", "regime", "mode", "channels_per_band", "conflict_vs_degraded",
        "local", "global_mean", "conflict_vs_wrong", "acc_in_cohort", "acc_external", "loc_hit",
        "loc_chance", "t_audit_s"]
print(grid.reindex(columns=cols).round(3).to_string(index=False))
print(f"\nelapsed {time.time() - t0:.0f} s")

/tmp/ipykernel_471/3797905100.py:111: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.504 against the supplied orientation data - a margin of 0.004 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


 n_bands topology   scale regime  mode  channels_per_band  conflict_vs_degraded  local  global_mean  conflict_vs_wrong  acc_in_cohort  acc_external  loc_hit  loc_chance  t_audit_s
       3 complete     SNV     H1 equal                329                 0.720  0.864        0.684              0.733          0.905         0.839    0.429       0.429      0.691
       5 complete     SNV     H1 equal                197                 0.746  0.877        0.711              0.672          0.894         0.814    0.548       0.314      1.900
       7 complete     SNV     H1 equal                141                 0.777  0.924        0.738              0.648          0.877         0.797    0.619       0.143      0.927
      10 complete     SNV     H1 equal                 98                 0.771  0.889        0.739              0.507          0.882         0.785    0.786       0.229      1.625
      14 complete     SNV     H1 equal                 70                 0.762  0.856        0.736 

## 2 · Five patient splits for the configurations the guide compares

Mean ± sd over `SPLIT_REPEAT = 0–4`, and each configuration against the default (7 bands, complete, SNV) *within* each split, which removes the split-to-split variation from the comparison.

In [4]:
t0 = time.time()
KEY = [dict(n_bands=3), dict(n_bands=5), dict(n_bands=7), dict(n_bands=10), dict(n_bands=14),
       dict(n_bands=7, topology="chain"), dict(n_bands=10, topology="chain"),
       dict(n_bands=7, topology="knn"), dict(n_bands=7, scale="min-max")]
splits = [sp0] + [make_split(r) for r in range(1, 5)]
five = pd.DataFrame([evaluate(sp, **g) for sp in splits for g in KEY])
five["config"] = five.apply(lambda r: f"{r.n_bands} bands, {r.topology}, {r.scale}", axis=1)
five["lift"] = five["loc_hit"] / five["loc_chance"]
m = ["conflict_vs_degraded", "local", "global_mean", "conflict_vs_wrong", "acc_in_cohort", "acc_external", "loc_hit", "lift"]
g = five.groupby("config", sort=False)[m]
summary = g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)
print(summary.to_string())

base = five[five.config == "7 bands, complete, SNV"].set_index("repeat")
paired = []
for cfg, grp in five.groupby("config", sort=False):
    d = grp.set_index("repeat")["conflict_vs_degraded"] - base["conflict_vs_degraded"]
    paired.append({"config": cfg, "d_conflict": d.mean(), "sd": d.std(), "better_in": f"{(d > 0).sum()}/5",
                   "d_loc_hit": (grp.set_index("repeat")["loc_hit"] - base["loc_hit"]).mean(),
                   "d_acc_external": (grp.set_index("repeat")["acc_external"] - base["acc_external"]).mean()})
print("\nagainst the default, within each split:")
print(pd.DataFrame(paired).round(3).to_string(index=False))
print(f"\nelapsed {time.time() - t0:.0f} s")

/tmp/ipykernel_471/3797905100.py:111: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.504 against the supplied orientation data - a margin of 0.004 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_471/3797905100.py:111: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.501 against the supplied orientation data - a margin of 0.001 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_471/3797905100.py:111: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.491 against the supplied orientation data - a margin of 0.009 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_471/3797905100.py:111: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.524 against the supplied orientation data - a margin of 0.024 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


/tmp/ipykernel_471/3797905100.py:111: WeakOrientationWarning: the orientation of 'dg' was decided on an AUROC of 0.504 against the supplied orientation data - a margin of 0.004 from chance, so the sign is close to arbitrary. Getting it backwards turns a descriptor that separates at p into one that separates at 1-p. If the event you want flagged is not the supplied orientation data, pass orient_on= with labels for the event you mean.
  tri = Triage(conflict_score="dg", review_rate=REVIEW_RATE, novelty_q=NOVELTY_Q,


                           conflict_vs_degraded          local    global_mean conflict_vs_wrong  acc_in_cohort   acc_external        loc_hit           lift
config                                                                                                                                                     
3 bands, complete, SNV            0.675 ± 0.038  0.819 ± 0.037  0.639 ± 0.038     0.672 ± 0.043    0.9 ± 0.014  0.782 ± 0.055  0.543 ± 0.134  1.267 ± 0.313
5 bands, complete, SNV            0.726 ± 0.033  0.842 ± 0.049  0.695 ± 0.029     0.638 ± 0.041  0.902 ± 0.016  0.772 ± 0.041  0.538 ± 0.071  1.712 ± 0.225
7 bands, complete, SNV             0.76 ± 0.028  0.877 ± 0.049  0.728 ± 0.023     0.583 ± 0.051  0.885 ± 0.015  0.771 ± 0.026   0.652 ± 0.09   4.567 ± 0.63
10 bands, complete, SNV           0.756 ± 0.036  0.855 ± 0.055  0.729 ± 0.032     0.532 ± 0.056  0.881 ± 0.016  0.767 ± 0.025  0.824 ± 0.087   3.604 ± 0.38
14 bands, complete, SNV           0.754 ± 0.033  0.842 ± 0.055  

## 3 · Band width against the width of the change

A local defect 50, 100, 200 or 400 cm⁻¹ wide, at a random position. Per spectrum, it is planted on half the qc spectra. Per batch, it is planted in 42 localisation trials. Three splits.

In [5]:
t0 = time.time()
WIDTHS = (50.0, 100.0, 200.0, 400.0)
lo0, hi0 = BAND_RANGE

def window(r, w):
    c = r.uniform(lo0 + w / 2, hi0 - w / 2)
    return c - w / 2, c + w / 2

wrows = []
for sp in splits[:3]:
    y_qc, MM_qc = y_all[sp["ix"]["qc"]], MM_all[sp["ix"]["qc"]]
    for k in (5, 7, 10, 14, 20):
        fm = fit_model(sp, n_bands=k)
        for w in WIDTHS:
            r = np.random.default_rng(1000 + int(w))
            Xd, lab = MM_qc.copy(), np.zeros(len(MM_qc), int)
            for i in r.permutation(len(Xd))[: len(Xd) // 2]:
                a, b = window(r, w)
                sel, s = (AXIS >= a) & (AXIS < b), r.uniform(0.3, 1.0)
                Xd[i, sel] = Xd[i, sel] * np.exp(-0.45 * s) + r.normal(0, 0.02 * s, int(sel.sum()))
                lab[i] = 1
            c = fm["triage"].assess(fm["prep"](Xd), y_qc).table["conflict"].to_numpy()
            trials = []
            for t in range(42):
                r2 = np.random.default_rng(7000 + 97 * t + int(w))
                a, b = window(r2, w)
                trials.append((a, b, r2))
            hit, chance = localise(sp, fm, trials=trials)
            wrows.append({"repeat": sp["repeat"], "n_bands": k, "band_cm": round((hi0 - lo0) / k, 1),
                          "defect_cm": w, "conflict_local": roc_auc_score(lab, c),
                          "loc_hit": hit, "loc_chance": chance})
width = pd.DataFrame(wrows)
width["lift"] = width["loc_hit"] / width["loc_chance"]
for col in ("conflict_local", "loc_hit", "lift"):
    print(col)
    print(width.pivot_table(index=["n_bands", "band_cm"], columns="defect_cm", values=col).round(3).to_string(), "\n")
print(f"elapsed {time.time() - t0:.0f} s")

conflict_local
defect_cm        50.0   100.0  200.0  400.0
n_bands band_cm                            
5       278.5    0.639  0.767  0.876  0.951
7       198.9    0.638  0.763  0.882  0.962
10      139.2    0.630  0.749  0.859  0.943
14      99.5     0.628  0.737  0.849  0.936
20      69.6     0.620  0.727  0.843  0.930 

loc_hit
defect_cm        50.0   100.0  200.0  400.0
n_bands band_cm                            
5       278.5    0.730  0.603  0.603  0.706
7       198.9    0.857  0.817  0.730  0.738
10      139.2    0.889  0.833  0.833  0.770
14      99.5     0.825  0.817  0.833  0.635
20      69.6     0.952  0.968  0.897  0.563 

lift
defect_cm         50.0   100.0  200.0  400.0
n_bands band_cm                             
5       278.5     3.129  2.303  1.689  1.469
7       198.9     4.941  4.006  2.556  1.722
10      139.2     6.222  4.795  3.302  1.925
14      99.5      7.956  5.722  3.858  1.769
20      69.6     10.811  8.053  4.650  1.655 

elapsed 716 s


## 4 · Coarser spectral sampling (split 0)

Adjacent channels are averaged *after* the defects are planted: the optics degrade the signal, then the detector samples it. The last row uses PLS-DA(5), because 9 channels per band cannot support 10 components.

In [6]:
t0 = time.time()
RES = [(1, 7, 10), (2, 7, 10), (4, 7, 10), (8, 7, 10), (8, 7, 5), (4, 14, 10), (8, 14, 5)]
res = []
for f, k, nc in RES:
    r = evaluate(sp0, n_bands=k, n_components=nc, bin_factor=f)
    r["step_cm"] = round(abs(float(np.diff(binf(AXIS, f)).mean())), 2)
    res.append(r)
res = pd.DataFrame(res)
print(res[["bin_factor", "step_cm", "n_bands", "n_components", "channels_per_band", "local",
           "conflict_vs_degraded", "loc_hit", "acc_external"]].round(3).to_string(index=False))
print(f"\nelapsed {time.time() - t0:.0f} s")

 bin_factor  step_cm  n_bands  n_components  channels_per_band  local  conflict_vs_degraded  loc_hit  acc_external
          1     1.41        7            10                141  0.924                 0.777    0.619         0.797
          2     2.82        7            10                 70  0.907                 0.718    0.571         0.799
          4     5.65        7            10                 35  0.880                 0.673    0.500         0.804
          8    11.29        7            10                 18  0.866                 0.651    0.619         0.782
          8    11.29        7             5                 18  0.851                 0.645    0.595         0.789
          4     5.65       14            10                 17  0.808                 0.644    0.738         0.806
          8    11.29       14             5                  9  0.795                 0.624    0.643         0.791

elapsed 120 s


## 5 · `peak_informed` returns the number of bands requested (fixed in 0.5.5)

In [7]:
for k in (5, 7, 10, 14):
    b = BandSet.peak_informed(AXIS, MM_all[sp0["ix"]["fit"]], k, lo=BAND_RANGE[0], hi=BAND_RANGE[1])
    print(f"requested {k:2d}, returned {len(b):2d}:", " ".join(f"{x.lo:.0f}" for x in b.bands), f"{b.bands[-1].hi:.0f}")

requested  5, returned  5: 400 690 973 1188 1500 1792
requested  7, returned  7: 400 690 838 973 1188 1328 1500 1792
requested 10, returned 10: 400 609 690 838 973 1188 1278 1424 1500 1594 1792
requested 14, returned 14: 400 609 690 838 907 973 1025 1120 1188 1278 1424 1500 1594 1741 1792


## 6 · Every value quoted in the guide and the response, checked

In [8]:
S5 = five.groupby("config", sort=False)
mean5 = lambda cfg, col: float(S5.get_group(cfg)[col].mean())  # noqa: E731
G = lambda **kw: grid.loc[np.logical_and.reduce([grid[k].astype(str) == str(v) for k, v in kw.items()])].iloc[0]  # noqa: E731
W = lambda k, w, col: float(width[(width.n_bands == k) & (width.defect_cm == w)][col].mean())  # noqa: E731
R = lambda f, k, nc: res[(res.bin_factor == f) & (res.n_bands == k) & (res.n_components == nc)].iloc[0]  # noqa: E731
claims = check([
    ("5 splits: 7 bands complete SNV",        mean5("7 bands, complete, SNV", "conflict_vs_degraded"), 0.760, 0.005),
    ("5 splits: sd of the above",             float(S5.get_group("7 bands, complete, SNV")["conflict_vs_degraded"].std()), 0.028, 0.005),
    ("5 splits: 7 bands min-max",             mean5("7 bands, complete, min-max", "conflict_vs_degraded"), 0.510, 0.005),
    ("5 splits: 7 bands chain",               mean5("7 bands, chain, SNV", "conflict_vs_degraded"), 0.715, 0.005),
    ("5 splits: 7 bands kNN",                 mean5("7 bands, knn, SNV", "conflict_vs_degraded"), 0.756, 0.005),
    ("5 splits: 3 bands",                     mean5("3 bands, complete, SNV", "conflict_vs_degraded"), 0.675, 0.005),
    ("5 splits: 5 bands",                     mean5("5 bands, complete, SNV", "conflict_vs_degraded"), 0.726, 0.005),
    ("5 splits: 10 bands",                    mean5("10 bands, complete, SNV", "conflict_vs_degraded"), 0.756, 0.005),
    ("5 splits: 14 bands",                    mean5("14 bands, complete, SNV", "conflict_vs_degraded"), 0.754, 0.005),
    ("5 splits: local, 7 complete",           mean5("7 bands, complete, SNV", "local"), 0.877, 0.005),
    ("5 splits: local, 7 chain",              mean5("7 bands, chain, SNV", "local"), 0.759, 0.005),
    ("5 splits: hit, 7 bands",                mean5("7 bands, complete, SNV", "loc_hit"), 0.652, 0.01),
    ("5 splits: hit, 10 bands",               mean5("10 bands, complete, SNV", "loc_hit"), 0.824, 0.01),
    ("5 splits: hit, 14 bands",               mean5("14 bands, complete, SNV", "loc_hit"), 0.819, 0.01),
    ("5 splits: hit, 7 kNN",                  mean5("7 bands, knn, SNV", "loc_hit"), 0.510, 0.01),
    ("5 splits: external acc, 7 bands",       mean5("7 bands, complete, SNV", "acc_external"), 0.771, 0.005),
    ("5 splits: external acc, 14 bands",      mean5("14 bands, complete, SNV", "acc_external"), 0.749, 0.005),
    ("split 0: external acc, 3 bands",        float(G(n_bands=3)["acc_external"]), 0.839, 0.005),
    ("split 0: external acc, 28 bands",       float(G(n_bands=28, topology="complete")["acc_external"]), 0.754, 0.005),
    ("split 0: regime plain",                 float(G(n_bands=7, regime="plain")["conflict_vs_degraded"]), 0.778, 0.005),
    ("split 0: regime H2",                    float(G(n_bands=7, regime="H2")["conflict_vs_degraded"]), 0.772, 0.005),
    ("split 0: peak 7, in-cohort acc",        float(G(n_bands=7, mode="peak")["acc_in_cohort"]), 0.918, 0.005),
    ("split 0: peak 7, external acc",         float(G(n_bands=7, mode="peak")["acc_external"]), 0.800, 0.005),
    ("split 0: peak 7, conflict",             float(G(n_bands=7, mode="peak")["conflict_vs_degraded"]), 0.753, 0.005),
    ("width: 7 bands, 50 cm-1",               W(7, 50, "conflict_local"), 0.638, 0.005),
    ("width: 7 bands, 400 cm-1",              W(7, 400, "conflict_local"), 0.962, 0.005),
    ("width: 20 bands, 50 cm-1",              W(20, 50, "conflict_local"), 0.620, 0.005),
    ("width: 20 bands, 400 cm-1",             W(20, 400, "conflict_local"), 0.930, 0.005),
    ("width: hit, 7 bands, 100 cm-1",         W(7, 100, "loc_hit"), 0.817, 0.01),
    ("width: hit, 20 bands, 100 cm-1",        W(20, 100, "loc_hit"), 0.968, 0.01),
    ("width: hit, 20 bands, 400 cm-1",        W(20, 400, "loc_hit"), 0.563, 0.01),
    ("sampling x1: local, 7 bands",           float(R(1, 7, 10)["local"]), 0.924, 0.005),
    ("sampling x8: local, 7 bands",           float(R(8, 7, 10)["local"]), 0.866, 0.005),
    ("sampling x8: local, 14 bands, PLS(5)",  float(R(8, 14, 5)["local"]), 0.795, 0.005),
])

                            quantity  this_run  quoted   tol   ok
      5 splits: 7 bands complete SNV    0.7596   0.760 0.005 True
           5 splits: sd of the above    0.0275   0.028 0.005 True
           5 splits: 7 bands min-max    0.5098   0.510 0.005 True
             5 splits: 7 bands chain    0.7148   0.715 0.005 True
               5 splits: 7 bands kNN    0.7560   0.756 0.005 True
                   5 splits: 3 bands    0.6753   0.675 0.005 True
                   5 splits: 5 bands    0.7257   0.726 0.005 True
                  5 splits: 10 bands    0.7563   0.756 0.005 True
                  5 splits: 14 bands    0.7539   0.754 0.005 True
         5 splits: local, 7 complete    0.8774   0.877 0.005 True
            5 splits: local, 7 chain    0.7585   0.759 0.005 True
              5 splits: hit, 7 bands    0.6524   0.652 0.010 True
             5 splits: hit, 10 bands    0.8238   0.824 0.010 True
             5 splits: hit, 14 bands    0.8190   0.819 0.010 True
          

In [9]:
grid.to_csv("parameter_grid_split0.csv", index=False)
five.to_csv("parameter_five_splits.csv", index=False)
width.to_csv("parameter_band_width.csv", index=False)
res.to_csv("parameter_sampling.csv", index=False)
claims.to_csv("parameter_claims.csv", index=False)
print("wrote 5 CSV files")

wrote 5 CSV files
